# AimSegDL full training pipeline

In [ ]:
import torch
from aimsegdl.training.config import TrainingConfig
from aimsegdl.training.train import train


### Data preparation

In [ ]:
from aimsegdl.data_preparation.data_preparation import split_dataset

split_dataset(
    in_root = "datasets_em",
    out_root = "prepared_data_em",
    create_test_split = True,
)

### Model training

In [ ]:
config = TrainingConfig(
    learning_rate = 1e-3,
    batch_size = 8,
    num_epochs = 2000,
    num_workers = 0,
    image_height = 512,
    image_width = 512,
    pin_memory = True,
    device = "cuda" if torch.cuda.is_available() else "cpu",
    fibre_threshold = 0.5,
    axon_threshold = 0.5,
    min_diameter = 30.0,
    train_dir = "prepared_data_em/train_tiles/",
    val_dir = "prepared_data_em/val_tiles/",
    pretrained_weights = None,     # Path to .pth weights file or model name (if available) to initialize model
    load_checkpoint = True,       # Resume from checkpoint
    model_name = "aimsegdl",
    )

In [ ]:
train(config)

### Model evaluation

In [ ]:
import torch
from aimsegdl.evaluation.evaluate_model import evaluate_model_on_testset

test_dir = "prepared_data_em/test_tiles/"
model_path = "best_weights_model.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

evaluate_model_on_testset(
    model_path,
    test_dir,
    device,
    output_csv = "Evaluation Results",
    show_plots = True
)

### Export bioimageio model

In [ ]:
from aimsegdl.export_utils.create_bioimageio_model import export_bioimageio

export_bioimageio(
    model_path = f"best_weights_model.pth",
    model_name = "aimsegdl",
    test_img_path = r"aimsegdl\example_images\P03A_Frame5_roi0.tif",
    output_dir = "bioimageio_model",
    model_pixel_size = 0.008
    )